# Crip_paly_scene_app

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 必要に応じて追加のライブラリをインストール
!pip install torch torchvision torchaudio
!pip install pandas numpy matplotlib tqdm opencv-python

# 必要なパッケージのインストール
!pip install -q opencv-python==4.10.0.84
!pip install -q opencv-contrib-python==4.10.0.84
!pip install -q ultralytics
!pip install -q numpy==1.23.5 --force-reinstall # numpyのバージョンを修正し、強制的に再インストール
!pip install -q matplotlib==3.9.0
!pip install -q tqdm==4.66.4

print("\n✓ パッケージのインストールが完了しました")

In [ ]:
import os
import sys

# プロジェクトのパスを指定（例: /content/drive/MyDrive/Visuable_for_you_tabletennis）
PROJECT_PATH = '/content/drive/MyDrive/Visuable_for_you_tabletennis'

# プロジェクトパスをPythonのパスに追加
sys.path.insert(0, PROJECT_PATH)

# 作業ディレクトリを変更
os.chdir(PROJECT_PATH)
print(f"作業ディレクトリ: {os.getcwd()}")

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import cv2
from pathlib import Path
import json
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from IPython.display import Video, display, HTML

# Task1
from src.detection.table_detector import TableDetector
from src.detection.yolopose_tracker import YOLOPose_Tracker
from src.detection.player_classifier import PlayerClassifier
from src.detection.data_classes import TableInfo, PersonTrack
from src.detection.tracking_exporter import TrackingExporter
from src.visualization.player_classifier_visualizer import PlayerClassifierVisualizer

# Task2
# プロジェクトのモジュールをインポート
from src.models.play_classifier import PlayClassifierLSTM
from src.dataset.dataset import PoseSequenceDataset, InMemoryPoseSequenceDataset


print("✓ モジュールのインポートが完了しました")

print("インポート完了")
print(f"PyTorchバージョン: {torch.__version__}")
print(f"OpenCVバージョン: {cv2.__version__}")
print(f"CUDAが利用可能: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
TABLE_MODEL_PATH = 'models/tabel_detection_models.pt'
POSE_MODEL_PATH = 'models/yolo11n-pose.pt'
LSTM_MODEL_PATH = 'models/lstm_model.pth'
LSTM_CONFIG_PATH = 'models/lstm_config.json'

# 処理パラメータ
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Task1: 動画からプレイヤーの骨格データの獲得
FPS = 30.0              # 処理FPS（GPU利用時は高FPS推奨）
MAX_PLAYERS = 4         # 最大プレイヤー数
MIN_PLAYER_SCORE = 0.3  # プレイヤー判定の最小スコア閾値（0.0-1.0）
MIN_CONSECUTIVE_FRAMES = 30  # CS最小連続フレーム数
MAX_FRAME_GAP = 5           # 連続性を判定する際の最大フレーム間隔

save_video = True  # 骨格データを描画した動画を保存するかどうか

print("処理パラメータ:")
print(f"  処理FPS: {FPS}")
print(f"  最大プレイヤー数: {MAX_PLAYERS}")
print(f"  最小スコア閾値: {MIN_PLAYER_SCORE}")
print(f"  最小連続フレーム数: {MIN_CONSECUTIVE_FRAMES}")
print(f"  最大フレーム間隔: {MAX_FRAME_GAP}")
print(f"  保存: {save_video}")

In [ ]:
INPUT_VIDEO = 'data/raw/sample_video_01_01.MOV'
OUTPUT_VIDEO = 'output/predictions/sample_video_01_01_classification.MOV'
OUTPUT_POSE_VIDEO = 'output/predictions/sample_video_01_01_poses.MOV'
CSV_OUTPUT = 'output/predictions/sample_video_01_01_poses.csv'

In [ ]:
# メイン処理
def run_player_classification():
    """プレイヤー分類テストを実行"""
    
    print(f"\n動画ファイルを開いています: {INPUT_VIDEO}...")
    cap = cv2.VideoCapture(INPUT_VIDEO)
    if not cap.isOpened():
        print("エラー: 動画ファイルを開けませんでした")
        return None, None, None
    print("✓ 動画ファイルを開きました")
    
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    video_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"\n入力情報:")
    print(f"  解像度: {width}x{height}")
    print(f"  動画FPS: {video_fps:.2f}")
    print(f"  総フレーム数: {total_frames}")
    print(f"  処理FPS: {FPS:.2f}")
    print(f"  出力動画FPS: {FPS:.2f} (処理したフレームのみ出力)")
    print(f"  保存モード: {'ファイルに保存' if save_video else 'メモリに保持'}\n")

    # フレーム間隔を計算（四捨五入で正確に）
    frame_interval = max(1, round(video_fps / FPS))
    
    print("コンポーネントを初期化しています...")
    table_detector = TableDetector(yolo_model_path=TABLE_MODEL_PATH)
    pose_tracker = YOLOPose_Tracker(model_path=POSE_MODEL_PATH, device = 'cuda')
    player_classifier = PlayerClassifier(max_players=MAX_PLAYERS,min_player_score=MIN_PLAYER_SCORE)
    
    # 常に初期化（メモリ保持のため）
    visualizer = PlayerClassifierVisualizer(table_detector, pose_tracker, player_classifier)
    exporter = TrackingExporter()
    
    # 動画保存は save_video=True の時のみ
    video_writer = None
    if save_video:
        # 出力ディレクトリが存在しない場合は作成
        output_dir = Path(OUTPUT_POSE_VIDEO).parent
        output_dir.mkdir(parents=True, exist_ok=True)

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        video_writer = cv2.VideoWriter(OUTPUT_POSE_VIDEO, fourcc, FPS, (width, height))
        if not video_writer.isOpened():
            print(f"警告: VideoWriterの初期化に失敗しました: {OUTPUT_POSE_VIDEO}")
            video_writer = None

    # 卓球台を検出
    print("卓球台を検出中...")
    table_info = None
    max_detection_attempts = 100
    for attempt in range(max_detection_attempts):
        ret, frame = cap.read()
        if not ret:
            print("エラー: 動画の終端に達しました")
            cap.release()
            if video_writer:
                video_writer.release()
            return None, None, None
        table_info = table_detector.detect_table_from_frame(frame, frame_idx=attempt, force_detect=True)
        if table_info is not None:
            print(f"✓ 卓球台を検出しました（フレーム {attempt + 1}、信頼度: {table_info.confidence:.2f}）\n")
            break
    if table_info is None:
        print(f"エラー: 卓球台を検出できませんでした")
        cap.release()
        if video_writer:
            video_writer.release()
        return None, None, None

    # 動画を最初に戻す
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    frame_count = 0
    processed_count = 0
    player_ids = set()
    print("処理開始...\n")
    print(f"  フレーム間隔: {frame_interval} ({frame_interval}フレームごとに1回処理)")
    print(f"  予測処理フレーム数: 約{total_frames // frame_interval}フレーム")
    print(f"  処理率: {100.0 / frame_interval:.1f}%\n")

    # プログレスバー付きで処理
    pbar = tqdm(total=total_frames, desc="Processing")
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame_count += 1
            pbar.update(1)
            if frame_count % frame_interval != 0:
                continue
            processed_count += 1
            
            persons = pose_tracker.track_frame_with_table_filter(frame, table_info)
            if table_info and persons:
                player_classifier.update(persons, table_info, frame_count)

            if table_info:
                selected_ids, removed_ids = player_classifier.classify_players()
                player_ids = set(selected_ids)
                if removed_ids:
                    pose_tracker.remove_validated_track_ids(removed_ids)

            # プレイヤーの骨格データを常にメモリに保持
            if player_ids:
                player_persons = [p for p in persons if p.track_id in player_ids]
                if player_persons:
                    timestamp = frame_count / video_fps
                    exporter.add_frame(frame_count, timestamp, player_persons)

            # 動画保存は save_video=True の時のみ描画処理を実行
            if video_writer:
                # 結果を描画
                display_frame = visualizer.draw_results(frame, table_info, persons, player_ids)
                display_frame = visualizer.draw_candidate_info(display_frame, player_ids)
                cv2.putText(
                    display_frame,
                    f"Frame: {frame_count}/{total_frames} (Processed: {processed_count})",
                    (10, height - 20),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (255, 255, 255),
                    2
                )
                cv2.putText(
                    display_frame,
                    f"Detected: {len(persons)} persons, Players: {len(player_ids)}",
                    (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (255, 255, 255),
                    2
                )
                video_writer.write(display_frame)

    finally:
        pbar.close()
        cap.release()
        if video_writer:
            video_writer.release()

        print(f"\n✓ 処理完了:")
        print(f"  処理フレーム数: {frame_count}")
        print(f"  実際に処理したフレーム数: {processed_count}")
        print(f"  検出されたプレイヤーID: {sorted(player_ids)}")
        print(f"  候補者数: {len(player_classifier.candidates)}")

        if player_classifier.candidates:
            print(f"\n=== 候補者詳細 ===")
            candidates = []
            for track_id, candidate in player_classifier.candidates.items():
                if candidate.total_frames >= player_classifier.min_tracking_frames:
                    score = player_classifier._calculate_player_score(candidate)
                    candidates.append((track_id, candidate, score))
            candidates.sort(key=lambda x: x[2], reverse=True)

            for track_id, candidate, score in candidates:
                is_player = track_id in player_ids
                print(f"\nID {track_id} {'[PLAYER]' if is_player else ''}:")
                print(f"  スコア: {score:.3f}")
                print(f"  フレーム数: {candidate.total_frames}")
                print(f"  総運動量: {candidate.total_movement:.1f}")
                print(f"  卓球台付近比率: {candidate.near_table_ratio:.1%}")
        
        # 連続性フィルタリングを適用（常に実行）
        exporter.filter_by_consecutive_frames(
            min_consecutive_frames=MIN_CONSECUTIVE_FRAMES,
            max_frame_gap=MAX_FRAME_GAP
        )
        
        # 骨格データの正規化を実行（訓練データと同じ方法：腰幅ベース）
        exporter.normalize_poses(min_confidence=0.3)

        if save_video:
            print(f"\n出力ビデオ: {OUTPUT_POSE_VIDEO} ({FPS:.1f}fps)")
            player_roles = {track_id: "player" for track_id in player_ids}
            exporter.export_csv(CSV_OUTPUT, player_roles)
            print(f"プレイヤー骨格データをCSVに保存しました: {CSV_OUTPUT}")
        else:
            # メモリに保持
            print(f"\n出力ビデオ: メモリに保持")
            print(f"プレイヤー骨格データ: メモリに保持")
            print(f"  総フレーム数（フィルタ後）: {len(exporter.frame_data_list)}")

        # exporterとframe_interval, video_fpsを返す
        return exporter, frame_interval, video_fps

# テストを実行
exporter, frame_interval, video_fps = run_player_classification()

In [ ]:
# 設定ファイルを読み込み
if os.path.exists(LSTM_CONFIG_PATH):
    with open(LSTM_CONFIG_PATH, 'r') as f:
        config = json.load(f)
    print("設定ファイルを読み込みました:")
    for key, value in config.items():
        print(f"  {key}: {value}")
else:
    # デフォルト設定
    print("警告: 設定ファイルが見つかりません。デフォルト設定を使用します。")
    config = {
        'model_type': 'lstm',
        'hidden_size': 128,
        'num_layers': 2,
        'dropout': 0.3,
        'no_attention': False,
        'sequence_length': 30
    }

# モデル作成
print("\nモデルを作成中...")
model = PlayClassifierLSTM(
    input_size=34,
    hidden_size=config['hidden_size'],
    num_layers=config['num_layers'],
    dropout=config['dropout'],
    use_attention=not config.get('no_attention', False)
)

# 重み読み込み
checkpoint = torch.load(LSTM_MODEL_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(DEVICE)
model.eval()

print(f"モデル読み込み完了: {LSTM_MODEL_PATH}")
print(f"エポック: {checkpoint.get('epoch', 'N/A')}")
print(f"Best Val F1: {checkpoint.get('best_val_f1', 'N/A')}")

In [ ]:
# exporterからデータセット作成用のデータを取得
print("\nInMemoryPoseSequenceDataset作成中...")
# 全てのトラッキングIDを取得
all_track_ids = exporter.get_all_track_ids()
print(f"検出されたトラッキングID: {all_track_ids}")

# 全データを取得（track_id指定なし）
pose_data, frames = exporter.get_pose_data_for_dataset()
print(f"\nデータ形状:")
print(f"  pose_data: {pose_data.shape}")  # (num_frames, 34)
print(f"  frames: {frames.shape}")  # (num_frames,)

# InMemoryPoseSequenceDatasetを作成
sequence_length = config.get('sequence_length', 30)
# 訓練時の正規化パラメータを使用（checkpointに保存されている場合）
feature_mean = checkpoint.get('feature_mean', None)
feature_std = checkpoint.get('feature_std', None)

dataset = InMemoryPoseSequenceDataset(
    pose_data=pose_data,
    frames=frames,
    labels=None,  # 予測のみなのでラベルなし
    sequence_length=sequence_length,
    stride=1,  # 全フレームを予測
    normalize_features=False,
    feature_mean=feature_mean,
    feature_std=feature_std
)

print(f"\nInMemoryPoseSequenceDataset作成完了:")
print(f"  総フレーム数: {len(pose_data)}")
print(f"  シーケンス数: {len(dataset)}")
print(f"  シーケンス長: {sequence_length}フレーム")
print(f"  特徴量次元: {pose_data.shape[1]}")

THRESHOLD = 0.5

# 予測実行
print(f"\n予測開始 (閾値: {THRESHOLD})...")
frame_probs = {}  # {frame_num: [probs]}

with torch.no_grad():
    for features, _, metadata in tqdm(dataset, desc="予測中"):
        # バッチ次元を追加
        features = features.unsqueeze(0).to(DEVICE)
        
        # 予測
        outputs = model(features)  # (1, seq, 1)
        probs = outputs.squeeze().cpu().numpy()
        
        # フレームごとに確率を記録
        start_frame = metadata['start_frame']
        for i, prob in enumerate(probs):
            frame_num = start_frame + i
            if frame_num not in frame_probs:
                frame_probs[frame_num] = []
            frame_probs[frame_num].append(prob)

# 各フレームの確率を平均
predictions = []
for frame_num in sorted(frame_probs.keys()):
    avg_prob = np.mean(frame_probs[frame_num])
    prediction = 1 if avg_prob >= THRESHOLD else 0
    predictions.append({
        'frame': frame_num,
        'probability': avg_prob,
        'prediction': prediction,
        'num_predictions': len(frame_probs[frame_num])
    })

result_df = pd.DataFrame(predictions)

In [ ]:
MIN_SCENE_DURATION = 10  # 最小シーン長（フレーム数）

def extract_scenes(result_df, min_duration=MIN_SCENE_DURATION):
    """
    連続したプレー区間を抽出
    
    Args:
        result_df: 予測結果のDataFrame
        min_duration: 最小シーン長（フレーム数）
    
    Returns:
        [(start_frame, end_frame), ...] のリスト
    """
    scenes = []
    in_scene = False
    scene_start = None
    
    for _, row in result_df.iterrows():
        frame = row['frame']
        pred = row['prediction']
        
        if pred == 1:  # プレー中
            if not in_scene:
                scene_start = frame
                in_scene = True
        else:  # プレー外
            if in_scene:
                # シーン終了
                if frame - scene_start >= min_duration:
                    scenes.append((scene_start, frame - 1))
                in_scene = False
    
    # 最後のシーン
    if in_scene and result_df['frame'].iloc[-1] - scene_start >= min_duration:
        scenes.append((scene_start, result_df['frame'].iloc[-1]))
    
    return scenes

# シーン検出
scenes = extract_scenes(result_df, min_duration=MIN_SCENE_DURATION)

# 統計とプレー中時間の計算
num_play_frames = np.sum(result_df['prediction'] == 1)
play_ratio = num_play_frames / len(result_df) * 100 if len(result_df) > 0 else 0

# プレー中時間を計算（FPSから秒数を計算）
play_time_seconds = num_play_frames / FPS
play_time_minutes = play_time_seconds / 60
total_time_seconds = len(result_df) / FPS
total_time_minutes = total_time_seconds / 60

print(f"\n予測結果:")
print(f"  プレー中フレーム: {num_play_frames} / {len(result_df)} ({play_ratio:.1f}%)")
print(f"  プレー中時間: {play_time_minutes:.2f}分 ({play_time_seconds:.1f}秒)")
print(f"  総時間: {total_time_minutes:.2f}分 ({total_time_seconds:.1f}秒)")

print(f"\nシーン検出結果:")
print(f"  検出シーン数: {len(scenes)}")
print(f"  最小シーン長: {MIN_SCENE_DURATION}フレーム ({MIN_SCENE_DURATION/FPS:.1f}秒)")

# シーン情報を表示（最初の10シーン）
if scenes:
    print(f"\n主要シーン（最初の10シーン）:")
    for i, (start, end) in enumerate(scenes[:10], 1):
        duration_frames = end - start + 1
        duration_sec = duration_frames / FPS
        print(f"  シーン{i}: フレーム {start}-{end} ({duration_sec:.1f}秒, {duration_frames}フレーム)")

# 先頭10件を表示
print("\n予測結果（先頭10件）:")
display(result_df.head(10))

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path # 追加

# グラフの作成と保存
if save_video:
    print(f"\n予測グラフを作成中...")

    plt.figure(figsize=(16, 6))

    # 予測確率をプロット
    plt.subplot(2, 1, 1)
    plt.plot(result_df['frame'], result_df['probability'], linewidth=1, alpha=0.7, color='blue')
    plt.axhline(y=THRESHOLD, color='red', linestyle='--', label=f'閾値 ({THRESHOLD})')
    plt.fill_between(result_df['frame'], 0, result_df['probability'],
                        where=(result_df['probability'] >= THRESHOLD),
                        alpha=0.3, color='green', label='プレー中')
    plt.xlabel('フレーム番号')
    plt.ylabel('プレー中確率')
    plt.title('プレーシーン予測結果 - 確率')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.ylim(-0.05, 1.05)

    # 予測ラベルをプロット
    plt.subplot(2, 1, 2)
    plt.plot(result_df['frame'], result_df['prediction'], linewidth=1.5, color='green')
    plt.fill_between(result_df['frame'], 0, result_df['prediction'],
                        alpha=0.3, color='green')

    # 検出されたシーンを赤線でマーク
    for start, end in scenes:
        plt.axvline(x=start, color='red', linestyle=':', alpha=0.5, linewidth=1)
        plt.axvline(x=end, color='red', linestyle=':', alpha=0.5, linewidth=1)

    plt.xlabel('フレーム番号')
    plt.ylabel('予測ラベル (0: 非プレー, 1: プレー)')
    plt.title(f'プレーシーン予測結果 - 分類 (検出シーン数: {len(scenes)})')
    plt.grid(True, alpha=0.3)
    plt.ylim(-0.1, 1.1)
    plt.yticks([0, 1], ['非プレー', 'プレー'])

    plt.tight_layout()

    # グラフを保存
    # OUTPUT_VIDEOの拡張子が.MOVであるため、.mp4をreplaceしても意味がない。Pathlibを使って確実にpngにする。
    video_path_obj = Path(OUTPUT_VIDEO)
    base_name = video_path_obj.stem
    output_dir = video_path_obj.parent

    output_graph_path = output_dir / f"{base_name}_prediction_graph.png"
    plt.savefig(output_graph_path, dpi=150, bbox_inches='tight')
    print(f"予測グラフを保存しました: {output_graph_path}")
    plt.show()

    # 予測結果CSVを保存
    output_csv_path = output_dir / f"{base_name}_predictions.csv"
    result_df.to_csv(output_csv_path, index=False)
    print(f"予測結果CSVを保存しました: {output_csv_path}")

    # シーン情報をCSV保存
    if scenes:
        scenes_df = pd.DataFrame(scenes, columns=['start_frame', 'end_frame'])
        scenes_df['duration_frames'] = scenes_df['end_frame'] - scenes_df['start_frame'] + 1
        scenes_df['duration_sec'] = scenes_df['duration_frames'] / FPS
        scenes_df['start_time_sec'] = scenes_df['start_frame'] / FPS
        scenes_df['end_time_sec'] = scenes_df['end_frame'] / FPS

        scenes_csv_path = output_dir / f"{base_name}_scenes.csv"
        scenes_df.to_csv(scenes_csv_path, index=False)
        print(f"シーン情報CSVを保存しました: {scenes_csv_path}")

In [ ]:
# プレー中シーンのみを切り抜いた動画を作成
def create_play_scenes_video(input_video_path, scenes, output_path, fps, frame_interval):
    """
    プレー中と判定されたシーンのみを連結した動画を作成
    
    Args:
        input_video_path: 元の動画ファイルパス
        scenes: [(start_frame, end_frame), ...] シーンのリスト
        output_path: 出力動画ファイルパス
        fps: 出力動画のFPS
        frame_interval: 元動画での処理時のフレーム間隔
    """
    if not scenes:
        print("警告: 切り抜くシーンがありません")
        return
    
    print(f"\nプレー中シーンを切り抜いた動画を作成中...")
    print(f"  入力動画: {input_video_path}")
    print(f"  出力動画: {output_path}")
    print(f"  シーン数: {len(scenes)}")
    
    # 元の動画を開く
    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        print("エラー: 動画ファイルを開けませんでした")
        return
    
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    video_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    
    # 出力動画の設定
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    total_written_frames = 0
    
    # 各シーンを処理
    for scene_idx, (start_frame, end_frame) in enumerate(tqdm(scenes, desc="シーン処理中")):
        # start_frameとend_frameは処理済みフレーム番号なので、
        # 元動画のフレーム番号に変換（整数に変換）
        original_start = int(start_frame * frame_interval)
        original_end = int(end_frame * frame_interval)
        
        # シーンの開始位置にシーク
        cap.set(cv2.CAP_PROP_POS_FRAMES, original_start)
        
        # このシーンのフレームを読み込んで書き込む
        for frame_idx in range(original_start, original_end + 1, frame_interval):
            ret, frame = cap.read()
            if not ret:
                break
            
            # シーン番号と再生時間をオーバーレイ
            current_time = total_written_frames / fps
            cv2.putText(
                frame,
                f"Scene {scene_idx + 1}/{len(scenes)} | Time: {current_time:.1f}s",
                (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )
            
            out.write(frame)
            total_written_frames += 1
    
    # クリーンアップ
    cap.release()
    out.release()
    
    # 統計表示
    output_duration = total_written_frames / fps
    print(f"\n✓ 動画作成完了:")
    print(f"  出力フレーム数: {total_written_frames}")
    print(f"  出力時間: {output_duration:.1f}秒 ({output_duration/60:.2f}分)")
    print(f"  出力FPS: {fps}")
    print(f"  保存先: {output_path}")

# プレーシーン動画を作成
if scenes and save_video:
    play_scenes_output = OUTPUT_VIDEO.replace('.mp4', '_play_scenes_only.mp4')
    create_play_scenes_video(
        input_video_path=INPUT_VIDEO,
        scenes=scenes,
        output_path=play_scenes_output,
        fps=FPS,
        frame_interval=frame_interval
    )
    
    # Colabで動画を表示
    print("\n生成された動画のプレビュー:")
    display(Video(play_scenes_output, width=800))
elif not scenes:
    print("\n警告: プレー中のシーンが検出されませんでした")
elif not save_video:
    print("\n情報: save_video=Falseのため、動画は作成されません")